# Presentation figures: what the tracker measures, what it tells us, what it lets us test

Four figures for 10 Nov. Mouse 720762; PL5 LFP unless stated. Uses saved tracker results and the CSVs from
tracker_vs_static.ipynb; the pilot in figure 4 uses the tracked CA3, MOs5 and EEG ch13 signals.

In [1]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
REPO = Path("/Users/namanrudrakshi/Developer/dc/dc-allen-data-acquire")
sys.path.insert(0, str(REPO / "code" / "python"))
from src import motif_tracking as mt, session_data as sd
from src.channel_sweep import list_signals
import warnings; warnings.filterwarnings("ignore")
plt.rcParams.update({"font.size": 10.5, "axes.spines.top": False, "axes.spines.right": False, "font.family": "DejaVu Sans"})
NWB = REPO / "data" / "raw" / "720762-20240221-v001.nwb"
STORE = REPO / "data" / "processed" / "motif_tracking" / "720762"
OUT = REPO / "data" / "out" / "2026-11-10_presentation_figures"; OUT.mkdir(parents=True, exist_ok=True)
TVS = REPO / "data" / "out" / "2026-11-10_tracker_vs_static"
def save(fig, name): fig.savefig(OUT / f"{name}.png", dpi=220, bbox_inches="tight", facecolor="white")

res = mt.load(STORE / "lfp_PL5.pkl")
io, nwb = sd.open_nwb(NWB); SRC = {s["key"]: tuple(s["source"]) for s in list_signals(nwb)}; io.close()
SIG = {k: mt.prepare(NWB, k, SRC[k], res.config) for k in ["lfp_PL5", "lfp_CA3", "lfp_MOs5", "eeg13"]}
sig = SIG["lfp_PL5"]
info, (run_t, run), (pupil_t, pupil) = mt.behaviour(NWB)
t, Zg = sig.t, sig.Z
tm = t / 60
K = len(res.library)
U = np.array([m.template for m in res.library])
E = res.second[[f"expr_M{i + 1}" for i in range(K)]].to_numpy()
Bm = res.second[[f"base_M{i + 1}" for i in range(K)]].to_numpy()
iso = np.interp(t, info.iso_t, info.iso_level)
okp = np.isfinite(pupil); pup = np.interp(t, pupil_t[okp], pupil[okp])
state = sd.state_of_time(t, info)
SPAN = {"awake": "#DCE9F5", "induction": "#E3E3E3", "isoflurane": "#F6DEDC"}
C = {"plus": "#C2504A", "minus": "#3E7CB8", "base": "#1D232B", "M1": "#1D232B", "M2": "#E0A030", "M3": "#2F7F8F"}

def shade(ax, strip=False):
    for a, b, c in [(info.t_start, info.iso_on, "awake"), (info.iso_on, info.induction_end, "induction"), (info.induction_end, info.iso_off, "isoflurane")]:
        ax.axvspan(a / 60, b / 60, color=SPAN[c], lw=0, zorder=0)
    if strip:
        for blocks, col in ((info.stim_blocks, "#E8955A"), (info.visual_blocks, "#6CB8A8")):
            for _, a, b in blocks:
                ax.axvspan(a / 60, b / 60, ymin=0.95, ymax=1.0, color=col, lw=0, zorder=4)

def smooth(v, n):
    return pd.Series(v).rolling(n, center=True, min_periods=1).mean().to_numpy()

def seg_rolling_sd(x, n=300):
    out = np.full(x.size, np.nan)
    for s in np.unique(sig.segment):
        i = np.flatnonzero(sig.segment == s)
        out[i] = pd.Series(x[i]).rolling(n, center=True, min_periods=min(n // 4, max(10, i.size // 2))).std().to_numpy()
    return out

def label(ax, letter, title):
    ax.set_title(f"{letter}   {title}", loc="left", fontsize=11.5, fontweight="bold")

## Figure 1: what the tracker measures, compared with the original representation
Same data (PL5, the broadband motif axis M1) shown three ways over the whole session.

In [2]:
u = U[0]
raw = Zg @ u
base = Bm[:, 0]
spread = seg_rolling_sd(raw - base)
fig, axes = plt.subplots(4, 1, figsize=(15, 10.5), sharex=True, gridspec_kw={"height_ratios": [2.4, 0.7, 1.6, 1.5]})

ax = axes[0]; shade(ax, strip=True)
ax.plot(tm, smooth(raw, 10), color="#9AA0A6", lw=0.6, label="activity along the motif axis (10-s mean)")
ax.fill_between(tm, base - spread, base + spread, color=C["base"], alpha=0.15, lw=0, label="spread (±1 SD around the baseline)")
ax.plot(tm, base, color=C["base"], lw=2.2, label="baseline (5-min centre)")
ax.set_ylabel("broadband motif axis\n(session-wide units)")
ax.legend(fontsize=9, frameon=False, loc="lower left", ncol=3)
label(ax, "A", "The raw signal along one motif: the centre moves, the spread changes")
for x_, txt in ((27, "awake"), (56, "induction"), (88, "isoflurane")):
    ax.text(x_, ax.get_ylim()[1] * 0.78, txt, ha="center", va="top", fontsize=10.5, color="#555555", fontweight="bold")

ax = axes[1]
bar = np.full(t.size, np.nan)
for st, (a, b) in sd.analysis_windows(info).items():
    if st == "full":
        continue
    m = (t >= a) & (t < b)
    Zs = Zg[m]; Zs = (Zs - Zs.mean(0)) / Zs.std(0)
    bar[m] = np.sign(Zs @ u)
ax.imshow(bar[None, :], aspect="auto", cmap=plt.matplotlib.colors.ListedColormap([C["minus"], C["plus"]]), vmin=-1, vmax=1,
          extent=[tm[0], tm[-1], 0, 1], interpolation="nearest")
ax.set_yticks([])
label(ax, "B", "Original representation: z-scored within each state, then + / − only (centre, spread and shape are discarded)")

ax = axes[2]; shade(ax)
ax.plot(tm, smooth(E[:, 0], 10), color="#555555", lw=0.6)
ax.axhline(0, color="k", lw=0.5)
ax.set_ylabel("expression\n(in units of spread)")
label(ax, "C", "Tracker expression: the fast switching between poles, measured against the moving centre")

ax = axes[3]; shade(ax)
ws = res.window_stats
P = ws.pivot_table(index="motif", columns="centre", values="present", aggfunc="first").reindex(range(K))
cent = P.columns.to_numpy() / 60
for i in range(K):
    on = P.iloc[i].to_numpy(bool)
    ax.scatter(cent[on], np.full(on.sum(), i), marker="s", s=14, color=[C["M1"], C["M2"], C["M3"]][i] if i < 3 else "#8A8F96", lw=0)
ax.set_yticks(range(K), [f"M{i + 1}" + (" broadband" if i == 0 else (" isoflurane-only" if i == 1 else "")) for i in range(K)], fontsize=8.5)
ax.set_ylim(K - 0.5, -0.5)
ax.set_xlabel("minutes")
label(ax, "D", "Tracker presence: which motifs exist in each 5-min window (directions are fixed; motifs appear and disappear)")
ax.set_xlim(tm[0], tm[-1])
fig.tight_layout(h_pad=1.2)
save(fig, "1_what_the_tracker_measures"); plt.show()

## Figure 2: the tracked quantities carry information the original discards

In [3]:
assoc = pd.read_csv(TVS / "1_state_association.csv")
pm1 = pd.read_csv(TVS / "1_pole_measures_Tracker_M1.csv")
fig = plt.figure(figsize=(15, 9.6))
gs = fig.add_gridspec(2, 2, width_ratios=[1.6, 1], hspace=0.42, wspace=0.22)

ax = fig.add_subplot(gs[0, 0]); shade(ax)
zb = lambda v: (v - np.nanmean(v)) / np.nanstd(v)
ax.plot(tm, zb(Bm[:, 0]), color=C["M1"], lw=2, label="M1 baseline (broadband)")
ax.plot(tm, zb(Bm[:, 1]), color=C["M2"], lw=2, label="M2 baseline (isoflurane motif)")
ax.set_ylabel("baseline position (z)")
ax2 = ax.twinx(); ax2.plot(info.iso_t / 60, info.iso_level, color="#777777", lw=1.2, ls="--"); ax2.set_ylabel("isoflurane (%)", color="#777777")
ax2.spines[["top"]].set_visible(False)
ax.legend(fontsize=9, frameon=False, loc="upper left"); ax.set_xlabel("minutes"); ax.set_xlim(tm[0], tm[-1])
label(ax, "A", "Baselines step with state (dashed: isoflurane %)")

ax = fig.add_subplot(gs[0, 1])
ws = res.window_stats.copy(); ws["state"] = sd.state_of_time(ws.centre.to_numpy(), info)
pres = ws.pivot_table(index="motif", columns="state", values="present", aggfunc="mean").reindex(columns=["awake", "induction", "isoflurane"])
ax.imshow(pres.to_numpy(), cmap="Greens", vmin=0, vmax=1, aspect="auto")
for i in range(pres.shape[0]):
    for j in range(3):
        v = pres.iat[i, j]; ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=9, color="white" if v > 0.6 else "k")
ax.set_xticks(range(3), ["awake", "induction", "isoflurane"]); ax.set_yticks(range(K), [f"M{i + 1}" for i in range(K)])
ax.spines[["left", "bottom"]].set_visible(False)
label(ax, "B", "Motifs appear and disappear with state")
ax.text(0, K + 0.35, "Directions stay put: M1 within 12° of its library direction\n(noise floor 7°); M2 exists only from induction on.",
        fontsize=9, va="top", ha="left", transform=ax.transData)

ax = fig.add_subplot(gs[1, 0])
iw = pm1[pm1.state == "isoflurane"].copy()
x_ = iw.centre / 60
ax.plot(x_, zb(iw.minus.to_numpy()), color=C["minus"], lw=2, label="− pole of M1 (5-min windows)")
ax.plot(x_, zb(iw.pupil.to_numpy()), color="#8E5EA2", lw=2, label="pupil size")
r = np.corrcoef(iw.minus, iw.pupil)[0, 1]
ax.set_ylabel("z within isoflurane"); ax.set_xlabel("minutes"); ax.legend(fontsize=9, frameon=False, loc="upper left")
label(ax, "C", f"Under isoflurane, the − pole tracks pupil (r = {r:.2f}); the + pole much less")
ax.text(0.99, 0.03, "holds for 2-, 5- and 10-min windows and beats a 60-s block shuffle", transform=ax.transAxes, ha="right", fontsize=8.5, color="#555555")

ax = fig.add_subplot(gs[1, 1])
TG = ["Isoflurane level (within maintenance)", "Pupil (within awake)", "Pupil (within maintenance)"]
TGL = ["dose\n(within isoflurane)", "pupil\n(awake)", "pupil\n(isoflurane)"]
MS = [("Original", "#9AA5B1", "original motif"), ("Relaxed", "#2F7F8F", "relaxed motifs"), ("Tracker expression", "#E0A030", "tracker expression"),
      ("Tracker baseline", "#C2504A", "tracker baseline"), ("Tracker spread", "#8E3B2E", "tracker spread")]
w = 0.16
for j, (mname, col, lab) in enumerate(MS):
    for i, tg in enumerate(TG):
        rr = assoc[(assoc.target == tg) & (assoc.method == mname)].iloc[0]
        ax.bar(i + (j - 2) * w, rr["|r|"], w * 0.92, color=col if rr.p <= 0.05 else "white", edgecolor=col, lw=1.3, label=lab if i == 0 else None)
ax.set_xticks(range(3), TGL); ax.set_ylabel("strongest correlation, |r|"); ax.set_ylim(0, 0.85)
ax.legend(fontsize=8.5, frameon=False, ncol=2, loc="upper left", title="hollow = not above chance", title_fontsize=8.5)
label(ax, "D", "Within-state information sits in the baseline")
fig.text(0.01, 0.005, "Checks: associations hold for baselines of 1–20 min; M1 and M2 are re-found in every re-run with other windows (|cos| ≥ 0.94). "
         "Caveat: dose changes under isoflurane coincide with an electrical-stimulation block.", fontsize=8.8, color="#555555")
save(fig, "2_what_the_tracked_quantities_tell_us"); plt.show()

## Figure 3: new hypotheses each tracker output makes testable

In [4]:
fig, ax = plt.subplots(figsize=(15, 8.6)); ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis("off")
OUTS = [("Expression", "fast switching between poles", 84, C["M2"]),
        ("Baseline", "where the motif's centre sits", 62, C["base"]),
        ("Spread & pole shape", "how far apart the poles are;\nwhether one pole moves alone", 40, C["minus"]),
        ("Presence & shared basis", "when motifs appear / disappear;\none library across states and signals", 17, C["M3"])]
def box(x, y, w, h, txt, sub, col, fs=11):
    ax.add_patch(FancyBboxPatch((x, y - h / 2), w, h, boxstyle="round,pad=0.4,rounding_size=1.5", fc="white", ec=col, lw=2))
    ax.text(x + 1.2, y + (1.6 if sub else 0), txt, fontsize=fs, fontweight="bold", va="center", color=col)
    if sub: ax.text(x + 1.2, y - 2.4, sub, fontsize=8.8, va="center", color="#444444")
for name, sub, y, col in OUTS:
    box(1, y, 21, 12, name, sub, col)
ax.text(1, 97, "Tracker outputs", fontsize=13, fontweight="bold")
ax.text(32, 97, "Hypotheses they make testable", fontsize=13, fontweight="bold")
STATUS = {"pilot": ("#2E7D32", "first pilot: figure 4"), "partial": ("#E0A030", "partial evidence"), "new": ("#777777", "untested")}
GROUPS = [("EEG ↔ LFP", 91, [("The EEG carries the LFP's baseline and spread more than its fast switching", ["Baseline", "Expression"], "pilot"),
                             ("EEG and LFP share one motif basis; the EEG shows a subset", ["Presence & shared basis"], "pilot"),
                             ("Motifs appear in deep LFP before the EEG at state transitions", ["Presence & shared basis"], "new")]),
          ("Opponent neurons", 59, [("Fast switching drives A/B alternation; the baseline drives slow A/B balance", ["Expression", "Baseline"], "partial"),
                                    ("Wider pole spread → stronger opponency between groups A and B", ["Spread & pole shape"], "new"),
                                    ("Each pole has its own neurons: when only the − pole moves, only group B changes", ["Spread & pole shape"], "new"),
                                    ("A motif appearing = a neuronal ensemble forming", ["Presence & shared basis"], "new")]),
          ("States & behaviour", 21, [("A state is a position (baseline, spread, active motifs) on a fixed basis", ["Baseline", "Spread & pole shape", "Presence & shared basis"], "partial"),
                                      ("Arousal (pupil) reshapes poles rather than switching motifs", ["Spread & pole shape"], "partial")])]
YOUT = {n: y for n, _, y, _ in OUTS}; COUT = {n: c for n, _, _, c in OUTS}
for gname, gy, hyps in GROUPS:
    ax.text(32, gy, gname, fontsize=12, fontweight="bold", va="center")
    for k, (txt, srcs, stt) in enumerate(hyps):
        y = gy - 6 - k * 6.2
        col, slab = STATUS[stt]
        ax.add_patch(FancyBboxPatch((32, y - 2.4), 52, 4.8, boxstyle="round,pad=0.25,rounding_size=1", fc="#FAFAFA", ec="#CCCCCC", lw=1))
        ax.text(33, y, txt, fontsize=9.6, va="center")
        ax.text(86, y, slab, fontsize=8.6, va="center", color=col, fontweight="bold")
        for s in srcs:
            ax.annotate("", xy=(31.6, y), xytext=(22.6, YOUT[s]), arrowprops=dict(arrowstyle="-|>", color=COUT[s], lw=0.9, alpha=0.55,
                                                                                    connectionstyle="arc3,rad=0.0", shrinkA=4, shrinkB=2))
save(fig, "3_new_hypotheses"); plt.show()

## Figure 4: pilot of the EEG ↔ LFP direction
Every signal is projected on the same PL5 motif directions (shared coordinates). Component coupling = correlation
between PL5 and the other signal, per state; baseline and spread with each condition segment's mean removed.

In [5]:
seg = sig.segment
def dm(v, m):
    v = v[m].copy(); s_ = seg[m]
    for x in np.unique(s_): v[s_ == x] -= v[s_ == x].mean()
    return v
def raw_spread(g, u):
    out = np.full(t.size, np.nan); x = (g.Z - g.base) @ u
    for s in np.unique(seg):
        i = np.flatnonzero(seg == s)
        out[i] = pd.Series(x[i]).rolling(300, center=True, min_periods=min(75, max(10, i.size // 2))).std().to_numpy()
    return out
OTH = {"lfp_CA3": "CA3", "lfp_MOs5": "MOs5", "eeg13": "EEG"}
rows = []
for k, lab in OTH.items():
    g = SIG[k]
    for mi in (0, 1):
        uu = U[mi]
        sp0, sp1 = raw_spread(sig, uu), raw_spread(g, uu)
        for st in ("awake", "isoflurane"):
            m = (state == st) & np.isfinite(sig.resid).all(1) & np.isfinite(g.resid).all(1) & np.isfinite(sp0) & np.isfinite(sp1)
            rows.append({"signal": lab, "motif": f"M{mi + 1}", "state": st,
                         "baseline": np.corrcoef(dm(sig.base @ uu, m), dm(g.base @ uu, m))[0, 1],
                         "spread": np.corrcoef(dm(sp0, m), dm(sp1, m))[0, 1],
                         "expression": np.corrcoef(sig.resid[m] @ uu, g.resid[m] @ uu)[0, 1]})
cpl = pd.DataFrame(rows); (OUT / "data").mkdir(exist_ok=True); cpl.to_csv(OUT / "data" / "4_component_coupling.csv", index=False)
basis = pd.DataFrame({lab: [np.abs(U[i] @ np.array([m.template for m in mt.load(STORE / f"{k}.pkl").library]).T).max() for i in range(3)]
                      for k, lab in OTH.items()}, index=["M1 (broadband)", "M2 (isoflurane)", "M3"])
basis.to_csv(OUT / "data" / "4_shared_basis.csv")
cpl.round(2)

,signal,motif,state,baseline,spread,expression
0,CA3,M1,awake,0.70,-0.24,0.14
1,CA3,M1,isoflurane,0.92,0.94,0.79
2,CA3,M2,awake,0.60,0.26,0.11
3,CA3,M2,isoflurane,0.53,0.80,0.38
4,MOs5,M1,awake,0.97,0.82,0.83
5,MOs5,M1,isoflurane,1.00,0.99,0.97
6,MOs5,M2,awake,0.78,0.68,0.66
7,MOs5,M2,isoflurane,0.91,0.89,0.70
8,EEG,M1,awake,0.90,0.41,0.62
9,EEG,M1,isoflurane,0.99,0.99,0.94


In [6]:
fig = plt.figure(figsize=(15, 9.4))
gs = fig.add_gridspec(2, 3, width_ratios=[0.8, 1.2, 1.2], height_ratios=[1, 1], hspace=0.5, wspace=0.3)
ax = fig.add_subplot(gs[0, 0])
ax.imshow(basis.to_numpy(), cmap="Greens", vmin=0.3, vmax=1, aspect="auto")
for i in range(basis.shape[0]):
    for j in range(basis.shape[1]):
        v = basis.iat[i, j]; ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=10, color="white" if v > 0.8 else "k")
ax.set_xticks(range(3), ["CA3\nhippocampus", "MOs5\nmotor ctx", "EEG ch13\nskull"], fontsize=9); ax.set_yticks(range(3), basis.index, fontsize=9)
ax.spines[["left", "bottom"]].set_visible(False)
label(ax, "A", "Shared basis")
ax.text(0.5, -0.22, "best |cos| between each PL5 motif\nand the other signal's own library", fontsize=8.5, va="top", ha="center", color="#555555", transform=ax.transAxes)

COMP = [("baseline", "#1D232B"), ("spread", "#3E7CB8"), ("expression", "#E0A030")]
for c_, mo in enumerate(["M1", "M2"]):
    ax = fig.add_subplot(gs[0, 1 + c_])
    groups = [(s, st) for s in OTH.values() for st in ("awake", "isoflurane")]
    w = 0.26
    for j, (comp, col) in enumerate(COMP):
        vals = [cpl[(cpl.signal == s) & (cpl.state == st) & (cpl.motif == mo)][comp].iloc[0] for s, st in groups]
        ax.bar(np.arange(len(groups)) + (j - 1) * w, vals, w * 0.9, color=col, label=comp)
    ax.set_xticks(range(len(groups)), [f"{s}\n{st}" for s, st in groups], fontsize=8.5)
    ax.set_ylim(-0.3, 1.05); ax.axhline(0, color="0.6", lw=0.6); ax.set_ylabel("correlation with PL5")
    for i in range(1, len(groups), 2): ax.axvspan(i - 0.5, i + 0.5, color="#F6DEDC", lw=0, zorder=0)
    label(ax, "B" if c_ == 0 else "C", f"{mo} {'(broadband)' if mo == 'M1' else '(isoflurane motif)'}: coupling with PL5")
    if c_ == 0:
        h, l = ax.get_legend_handles_labels()
        LEG = (h, l)
    if c_ == 1:
        ax.legend(*LEG, fontsize=9, frameon=False, ncol=3, loc="lower center")

ax = fig.add_subplot(gs[1, :])
aw = state == "awake"
for k, col, lab in (("lfp_PL5", "#1D232B", "PL5 LFP"), ("eeg13", "#C2504A", "EEG ch13"), ("lfp_CA3", "#2F7F8F", "CA3 LFP")):
    b = SIG[k].base @ U[0]
    bd = np.full(t.size, np.nan); okb = aw & np.isfinite(b); bd[okb] = dm(b, okb)
    bz = bd / np.nanstd(bd[aw])
    for sgi in np.unique(seg[aw]):
        i = aw & (seg == sgi)
        ax.plot(tm[i], bz[i], color=col, lw=2, label=f"{lab} baseline" if sgi == np.unique(seg[aw])[0] else None)
for sgi in np.unique(seg[aw])[1:]:
    ax.axvline(tm[seg == sgi][0], color="#BBBBBB", lw=0.8, ls=":")
x0 = SIG["lfp_PL5"].resid @ U[0]; x1 = SIG["eeg13"].resid @ U[0]
seg_a = (t >= 600) & (t < 720)
ax.set_ylabel("M1 baseline, awake\n(z, each segment's mean removed)"); ax.set_xlabel("minutes (awake period; dotted lines = condition boundaries)")
ax.legend(fontsize=9, frameon=False, loc="upper left", ncol=3)
r_pe = cpl[(cpl.signal == OTH["eeg13"]) & (cpl.state == "awake") & (cpl.motif == "M1")].iloc[0]
r_pc = cpl[(cpl.signal == OTH["lfp_CA3"]) & (cpl.state == "awake") & (cpl.motif == "M1")].iloc[0]
label(ax, "D", f"Awake: the EEG follows PL5's baseline (r = {r_pe.baseline:.2f}) better than its fast switching (r = {r_pe.expression:.2f}); "
               f"CA3 {r_pc.baseline:.2f} vs {r_pc.expression:.2f}")
fig.text(0.01, 0.005, "Pilot, one mouse, 4 signals. Pink columns = isoflurane, where everything couples. Caveats: baselines are smoothed, which raises "
         "correlations by itself (next: matched-timescale test and block-shuffle null); the awake EEG also carries a 2-Hz stepping artefact.",
         fontsize=8.8, color="#555555")
save(fig, "4_pilot_eeg_lfp_components"); plt.show()